In [2]:
import numpy as np
import pandas as pd
import re

In [ ]:
ois_excel = pd.read_excel('data/input/swap_ois_ea.xlsx', sheet_name='data_ESTR', skiprows=4)
ois_daily = pd.DataFrame()
ois_daily['Date'] = pd.to_datetime(ois_excel['Date'])
ois_daily['OIS_1M_n'] = ois_excel['EESWEA Curncy ']
ois_daily['OIS_2M_n'] = ois_excel['EESWEB Curncy']
ois_daily['OIS_3M_n'] = ois_excel['EESWEC Curncy']
ois_daily['OIS_4M_n'] = ois_excel['EESWED Curncy']
ois_daily['OIS_5M_n'] = ois_excel['EESWEE Curncy']
ois_daily['OIS_6M_n'] = ois_excel['EESWEF Curncy']
ois_daily['OIS_7M_n'] = ois_excel['EESWEG Curncy']
ois_daily['OIS_8M_n'] = ois_excel['EESWEH Curncy']
ois_daily['OIS_9M_n'] = ois_excel['EESWEI Curncy']
ois_daily['OIS_10M_n'] = ois_excel['EESWEJ Curncy']
ois_daily['OIS_11M_n'] = ois_excel['EESWEK Curncy']
ois_daily['OIS_1Y_n'] = ois_excel['EESWE1 Curncy']

ois_daily = ois_daily.dropna()
ois_daily

,Date,OIS_1M_n,OIS_2M_n,OIS_3M_n,OIS_4M_n,OIS_5M_n,OIS_6M_n,OIS_7M_n,OIS_8M_n,OIS_9M_n,OIS_10M_n,OIS_11M_n,OIS_1Y_n
525,2007-01-02,3.49000,3.51900,3.57900,3.62550,3.66450,3.70400,3.74550,3.77450,3.79800,3.82400,3.84450,3.86400
526,2007-01-03,3.49000,3.51900,3.57900,3.62900,3.66700,3.70600,3.74200,3.77150,3.79900,3.82300,3.84400,3.86500
527,2007-01-04,3.48850,3.51750,3.58650,3.62850,3.66850,3.70950,3.74450,3.77550,3.80150,3.82650,3.84750,3.86850
528,2007-01-05,3.48900,3.52400,3.59550,3.64000,3.68150,3.72150,3.75950,3.79350,3.82000,3.84000,3.86500,3.88500
529,2007-01-08,3.48550,3.52600,3.59500,3.64150,3.68050,3.72200,3.75900,3.79000,3.81800,3.84500,3.86500,3.88600
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5587,2026-05-28,2.03700,2.10335,2.14665,2.18575,2.22950,2.26250,2.29600,2.32200,2.34500,2.36650,2.38650,2.40250
5588,2026-05-29,2.04600,2.11035,2.15300,2.18575,2.22515,2.26105,2.28575,2.31145,2.33815,2.35170,2.36975,2.39325
5589,2026-06-01,2.05970,2.12355,2.16845,2.21910,2.26135,2.30220,2.33910,2.37230,2.40095,2.43200,2.45410,2.46930
5590,2026-06-02,2.07995,2.13620,2.17885,2.23000,2.27505,2.31525,2.35085,2.38505,2.41255,2.44025,2.46150,2.47845


In [7]:
#load the dataframes 

realised_inflation = pd.read_csv('data/input/HICP_monthly_YoY.csv', usecols=[0,2])
df_inflation = pd.DataFrame()
df_inflation['Date'] = realised_inflation['DATE']
df_inflation['pi_realized'] = realised_inflation['HICP Inflation rate - Total - Annual rate of change (HICP.M.U2.N.000000.4D0.ANR)']

realised_output = pd.read_csv('data/input/Industrial_Production_Monthly_wda_nsa.csv', usecols=[0,2])
df_output = pd.DataFrame()
df_output['Date'] = realised_output['DATE']
df_output['y_realized'] = realised_output['Total industrial production (excluding construction) - Working day adjusted, not seasonally adjusted (STBS.M.I10.W.PROD.NS0020.4D0.N.IX)']

In [70]:
realised_output = pd.read_csv('data/input/Economic_Sentiment_Indicator.csv', usecols=[0,2])
df_output = pd.DataFrame()
df_output['Date'] = realised_output['DATE']
df_output['y_realized'] = realised_output['Economic sentiment indicator (RTD.M.S0.S.Y_ESIND.F)']

In [78]:
realised_gdp = pd.read_csv('data/input/GDP_volume_Quarterly.csv', usecols=[0,2])
df_output = pd.DataFrame()
df_output['Date'] = realised_gdp['DATE']
df_output['y_realized'] = realised_gdp['Gross domestic product at market prices, volume (MNA.Q.Y.I10.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N)']

df_output

,Date,y_realized
0,1995-03-31,2113963.14
1,1995-06-30,2130670.79
2,1995-09-30,2135994.92
3,1995-12-31,2140682.92
4,1996-03-31,2144754.93
...,...,...
120,2025-03-31,3306314.30
121,2025-06-30,3310394.97
122,2025-09-30,3319134.12
123,2025-12-31,3324699.45


In [92]:
from tempdisagg import TempDisaggModel
import pandas as pd
import numpy as np


def mensualise_gdp_tempdisagg(df_quarterly, df_indicator=None,
                               col_q="y_realized", col_ind="indicator",
                               method="denton-cholette", verbose=True):
    """
    Parameters
    ----------
    df_quarterly  : DataFrame [Date (fin de trimestre), col_q]
    df_indicator  : DataFrame [Date (fin de mois), col_ind]  (requis si chow-lin)
    method        : "denton-cholette" | "chow-lin-opt" | "fernandez" | "ensemble"
    """
    df_q = df_quarterly.copy()
    df_q["Date"] = pd.to_datetime(df_q["Date"])
    df_q = df_q.sort_values("Date").reset_index(drop=True)

    # -----------------------------------------------------------------------
    # Construction du DataFrame long au format tempdisagg :
    #   Index = entier séquentiel (0, 1, 2, ...) — PAS une chaîne "1995Q1"
    #   Grain = 1, 2 ou 3  (mois dans le trimestre)
    #   y     = valeur trimestrielle répétée sur les 3 lignes du trimestre
    #   X     = indicateur mensuel (ou tendance neutre si absent)
    # -----------------------------------------------------------------------
    rows = []
    month_dates = []  # on garde les dates réelles pour reconstruire l'index final

    for i, row in df_q.iterrows():
        q  = pd.Period(row["Date"], "Q")
        m1 = q.asfreq("M", "S")       # premier mois du trimestre
        for grain in range(1, 4):
            month_ts = (m1 + (grain - 1)).to_timestamp("M")
            rows.append({
                "Index": i,            # entier pur : 0, 1, 2, ...
                "Grain": grain,        # 1, 2, 3
                "y":     row[col_q],
            })
            month_dates.append(month_ts)

    df_long = pd.DataFrame(rows)

    # Ajout de l'indicateur mensuel
    if df_indicator is not None:
        ind = (df_indicator.copy()
               .assign(Date=lambda d: pd.to_datetime(d["Date"])
                                       .dt.to_period("M")
                                       .dt.to_timestamp("M"))
               .rename(columns={col_ind: "X"})
               [["Date", "X"]])
        df_long["Date"] = month_dates
        df_long = df_long.merge(ind, on="Date", how="left").drop(columns="Date")
        if df_long["X"].isna().any():
            raise ValueError(
                f"L'indicateur a {df_long['X'].isna().sum()} valeurs manquantes. "
                "Compléter la série ou utiliser method='denton-cholette'."
            )
    else:
        # Colonne X obligatoire même sans indicateur (denton/fernandez l'ignore)
        df_long["X"] = np.arange(len(df_long), dtype=float)

    # Ordre exact attendu par tempdisagg
    df_long = df_long[["Index", "Grain", "y", "X"]].reset_index(drop=True)
    # -----------------------------------------------------------------------
    # Fit + predict
    # conversion="sum"     : PIB = flux cumulé sur le trimestre (standard)
    # conversion="average" : si ta série est une moyenne trimestrielle
    # -----------------------------------------------------------------------
    model = TempDisaggModel(method=method, conversion="sum")
    model.fit(df_long)

    if verbose:
        model.summary()

    y_hat = model.predict(full=False)

    if hasattr(y_hat, "ndim") and y_hat.ndim > 1:
        y_hat = y_hat.flatten()

    result = pd.DataFrame({
        "Date":       month_dates[:len(y_hat)],
        "y_realized": y_hat,
    }).sort_values("Date").reset_index(drop=True)


    # Vérification cohérence agrégation
    check   = (result
               .assign(Q=pd.to_datetime(result["Date"]).dt.to_period("Q"))
               .groupby("Q")["y_realized"].sum())
    ref     = df_q.assign(Q=pd.to_datetime(df_q["Date"]).dt.to_period("Q")
                          ).set_index("Q")[col_q]
    max_err = (check - ref).abs().max()
    if verbose:
        print(f"\n  Erreur d'agrégation max (doit être ~0) : {max_err:.4f}")

    return result

# =========================================================================
# USAGE
# =========================================================================

# Sans indicateur (recommandé pour commencer)
df_gdp_monthly = mensualise_gdp_tempdisagg(df_output, method="denton-cholette")

# Ensemble (teste tous les modèles et pondère par NNLS)
#df_gdp_monthly = mensualise_gdp_tempdisagg(df_output, method="ensemble")



Temporal Disaggregation Model Summary

Method: denton-cholette
No coefficients estimated.

  Erreur d'agrégation max (doit être ~0) : 0.0000


In [112]:
df_output = df_gdp_monthly.copy()
df_output['y_realized'] = np.log(df_output['y_realized'].astype(float))
df_output["y_realized"] = df_output["y_realized"] - df_output["y_realized"].shift(12)
df_output = df_output.dropna(subset=["y_realized"]).reset_index()

In [8]:
hicp_core_flash_surprises = pd.read_excel('data/input/Eurozone_Core_HICP_flash.xlsx')
hicp_core_flash_surprises = hicp_core_flash_surprises[
    hicp_core_flash_surprises['Period'].astype(str).str.strip().str.endswith('P', na=False)
].copy()


def parse_flash_date(x):
    if pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    if re.fullmatch(r'\d{4}-\d{2}-\d{2}(?:\s+00:00:00)?', s):
        s = s.replace(' 00:00:00', '')
        return pd.to_datetime(s, format='%Y-%d-%m', errors='coerce')
    if re.fullmatch(r'\d{1,2}/\d{1,2}/\d{4}(?:\s+00:00:00)?', s):
        s = s.replace(' 00:00:00', '')
        return pd.to_datetime(s, format='%m/%d/%Y', errors='coerce')
    if re.fullmatch(r'\d{4}-\d{2}-\d{2}T.*', s):
        return pd.to_datetime(s, errors='coerce')

    return pd.NaT

hicp_core_flash_surprises['raw_date'] = hicp_core_flash_surprises['Date'].astype(str).str.strip()
hicp_core_flash_surprises['date'] = hicp_core_flash_surprises['raw_date'].apply(parse_flash_date)
hicp_core_flash_surprises['date'] = hicp_core_flash_surprises['date'].dt.normalize()
bad_dates = hicp_core_flash_surprises.loc[
    hicp_core_flash_surprises['date'].isna(),
    ['Date', 'Period']
].copy()

print("Unparsed dates:")
print(bad_dates.head(20))

hicp_core_flash_surprises = hicp_core_flash_surprises.dropna(subset=['date']).copy()
hicp_core_flash_surprises = hicp_core_flash_surprises[
    (hicp_core_flash_surprises['date'] >= pd.Timestamp('2013-07-16')) &
    (hicp_core_flash_surprises['date'] <= pd.Timestamp('2026-06-30'))
].copy()

def to_float_pct(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x in {'-', '--', ''}:
        return np.nan
    return float(x.replace('%', ''))

hicp_core_flash_surprises['srv_med'] = hicp_core_flash_surprises['Srv Med'].apply(to_float_pct)
hicp_core_flash_surprises['actual']  = hicp_core_flash_surprises['Actual'].apply(to_float_pct)


hicp_core_flash_surprises['hicp_surprise'] = (
    hicp_core_flash_surprises['actual'] - hicp_core_flash_surprises['srv_med']
)
hicp_core_flash_surprises = hicp_core_flash_surprises.sort_values('date').reset_index(drop=True)
hicp_core_flash_surprises

Unparsed dates:
    Date Period
125  NaN  May P


,Date,Time,Period,Srv Med,Srv Avg,Srv High,Srv Low,Srv Num,Actual,Prior,Revised,raw_date,date,srv_med,actual,hicp_surprise
0,7/31/2013,11:00:00,Jul P,1.2%,1.2%,1.5%,0.9%,11,1.1%,1.2%,--,7/31/2013,2013-07-31,1.2,1.1,-0.1
1,8/30/2013,11:00:00,Aug P,1.1%,1.1%,1.2%,1.0%,18,1.1%,1.1%,--,8/30/2013,2013-08-30,1.1,1.1,0.0
2,9/30/2013,11:00:00,Sep P,1.1%,1.0%,1.1%,0.7%,19,1.0%,1.1%,--,9/30/2013,2013-09-30,1.1,1.0,-0.1
3,10/31/2013,11:00:00,Oct P,1.0%,1.0%,1.1%,1.0%,20,0.8%,1.0%,--,10/31/2013,2013-10-31,1.0,0.8,-0.2
4,11/29/2013,11:00:00,Nov P,0.9%,0.9%,1.0%,0.7%,17,1.0%,0.8%,--,11/29/2013,2013-11-29,0.9,1.0,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,2026-04-02 00:00:00,11:00:00,Jan P,2.3%,2.3%,2.6%,1.8%,40,2.2%,2.3%,--,2026-04-02 00:00:00,2026-02-04,2.3,2.2,-0.1
150,2026-03-03 00:00:00,11:00:00,Feb P,2.2%,2.2%,2.3%,2.0%,39,2.4%,2.2%,--,2026-03-03 00:00:00,2026-03-03,2.2,2.4,0.2
151,3/31/2026,11:00:00,Mar P,2.4%,2.4%,2.7%,2.2%,37,2.3%,2.4%,--,3/31/2026,2026-03-31,2.4,2.3,-0.1
152,4/30/2026,11:00:00,Apr P,2.2%,2.2%,2.9%,1.9%,41,2.2%,2.3%,--,4/30/2026,2026-04-30,2.2,2.2,0.0


In [113]:
PMI_surprise = pd.read_excel('data/input/PMI_EA_Composite.xlsx')

PMI_surprise['raw_date'] = PMI_surprise['MI'].astype(str).str.strip()
PMI_surprise['date'] = PMI_surprise['raw_date'].apply(parse_flash_date)
PMI_surprise['date'] = PMI_surprise['date'].dt.normalize()
bad_dates = PMI_surprise.loc[
    PMI_surprise['date'].isna(),
    ['MI', 'Period']
].copy()

print("Unparsed dates:")
print(bad_dates.head(20))

PMI_surprise = PMI_surprise.dropna(subset=['date']).copy()
PMI_surprise = PMI_surprise[
    (PMI_surprise['date'] >= pd.Timestamp('2013-07-16')) &
    (PMI_surprise['date'] <= pd.Timestamp('2026-06-30'))
].copy()

PMI_surprise = PMI_surprise.set_index('date')
PMI_surprise_prelim = PMI_surprise[::2]
PMI_surprise_prelim = PMI_surprise_prelim[:-1]
PMI_surprise_final = PMI_surprise[1::2]

PMI_surprise


Unparsed dates:
Empty DataFrame
Columns: [MI, Period]
Index: []


,MI,Time,Period,Srv Med,Srv Avg,Srv High,Srv Low,Srv Num,Actual,Prior,Revised,raw_date
date,,,,,,,,,,,,
2013-07-24,7/24/2013,10:00:00,Jul P,49.1,49.1,49.7,48.4,26,--,--,--,7/24/2013
2013-08-05,2013-05-08 00:00:00,10:00:00,Jul F,50.4,50.4,50.5,50.4,21,--,--,--,2013-05-08 00:00:00
2013-08-22,8/22/2013,10:00:00,Aug P,50.9,50.9,51.5,50.5,23,--,--,--,8/22/2013
2013-09-04,2013-04-09 00:00:00,10:00:00,Aug F,51.7,51.7,51.8,51.5,22,--,--,--,2013-04-09 00:00:00
2013-09-23,9/23/2013,10:00:00,Sep P,51.8,51.8,52.5,51.1,25,--,--,--,9/23/2013
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,4/23/2026,10:00:00,Apr P,50.1,50.2,51.0,49.5,28,--,50.7,--,4/23/2026
2026-05-06,2026-06-05 00:00:00,10:00:00,Apr F,48.6,48.6,49.1,48.6,19,48.8,50.7,--,2026-06-05 00:00:00
2026-05-21,5/21/2026,10:00:00,May P,48.8,48.8,49.5,47.5,28,--,48.8,--,5/21/2026


In [125]:
gdp_surprise = pd.read_excel('data/input/gdp_surv.xlsx')

gdp_surprise['raw_date'] = gdp_surprise['Date'].astype(str).str.strip()
gdp_surprise['date'] = gdp_surprise['raw_date'].apply(parse_flash_date)
gdp_surprise['date'] = gdp_surprise['date'].dt.normalize()
bad_dates = gdp_surprise.loc[
    gdp_surprise['date'].isna(),
    ['Date', 'Period']
].copy()

print("Unparsed dates:")
print(bad_dates.head(20))

gdp_surprise = gdp_surprise.dropna(subset=['date']).copy()
gdp_surprise = gdp_surprise[
    (gdp_surprise['date'] >= pd.Timestamp('2013-07-16')) &
    (gdp_surprise['date'] <= pd.Timestamp('2026-06-30'))
].copy()

gdp_surprise = gdp_surprise.set_index('date')


gdp_surprise

Unparsed dates:
Empty DataFrame
Columns: [Date, Period]
Index: []


,Date,Time,Period,Srv Med,Srv Avg,Srv High,Srv Low,Srv Num,Actual,Prior,Revised,raw_date
date,,,,,,,,,,,,
2013-08-14,8/14/2013,11:00:00,2Q A,-0.8%,-0.8%,-0.6%,-1.0%,28,-0.7%,-1.1%,--,8/14/2013
2013-09-04,2013-04-09 00:00:00,11:00:00,2Q P,-0.7%,-0.7%,-0.7%,-0.7%,26,-0.5%,-0.7%,-0.3%,2013-04-09 00:00:00
2013-11-14,11/14/2013,11:00:00,3Q A,-0.3%,-0.3%,-0.1%,-0.5%,28,-0.4%,-0.5%,-0.6%,11/14/2013
2013-12-04,2013-04-12 00:00:00,11:00:00,3Q P,-0.4%,-0.4%,-0.3%,-0.5%,28,-0.4%,-0.4%,0.2%,2013-04-12 00:00:00
2014-02-14,2/14/2014,11:00:00,4Q A,0.4%,0.4%,0.6%,0.3%,26,0.5%,-0.4%,-0.3%,2/14/2014
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-13,2/13/2026,11:00:00,4Q S,1.3%,1.3%,1.3%,1.3%,20,1.3%,1.3%,1.2%,2/13/2026
2026-03-06,2026-06-03 00:00:00,11:00:00,4Q T,1.3%,1.3%,1.3%,1.3%,21,1.2%,1.3%,--,2026-06-03 00:00:00
2026-04-30,4/30/2026,11:00:00,1Q A,0.9%,0.9%,1.0%,0.7%,21,0.8%,1.2%,--,4/30/2026


In [114]:
OIS_COLS = ['OIS_1M','OIS_3M','OIS_6M','OIS_1Y']
ois = pd.read_csv('data/input/HICP_core_surprises.csv', index_col=2)
df_ois = pd.DataFrame()
for col in OIS_COLS:
    df_ois[col] = ois[ois['instrument_label']==col]['variation']
flash_events = pd.to_datetime(hicp_core_flash_surprises['date']).dt.date
df_ois['Date'] = pd.to_datetime(df_ois.index,format ='%Y-%m-%d %H:%M:%S%z', utc=True).date
df_ois = df_ois[df_ois['Date'].isin(flash_events.values)]
df_ois['consensus'] = hicp_core_flash_surprises['srv_med'].values
df_ois['actual'] = hicp_core_flash_surprises['actual'].values
df_ois['indicator'] = 'HICP'
df_ois


,OIS_1M,OIS_3M,OIS_6M,OIS_1Y,Date,consensus,actual,indicator
event_datetime,,,,,,,,
2013-07-31 11:00:00+02:00,0.000,-0.10,-0.150,-0.300,2013-07-31,1.2,1.1,HICP
2013-08-30 11:00:00+02:00,0.000,-0.20,0.000,0.100,2013-08-30,1.1,1.1,HICP
2013-09-30 11:00:00+02:00,0.000,0.30,-0.400,-0.200,2013-09-30,1.1,1.0,HICP
2013-10-31 11:00:00+01:00,-0.050,-0.40,-0.900,-1.650,2013-10-31,1.0,0.8,HICP
2013-11-29 11:00:00+01:00,0.000,-0.10,-0.100,-0.200,2013-11-29,0.9,1.0,HICP
...,...,...,...,...,...,...,...,...
2026-02-04 11:00:00+01:00,0.000,0.19,0.165,0.245,2026-02-04,2.3,2.2,HICP
2026-03-03 11:00:00+01:00,0.100,0.30,0.650,1.550,2026-03-03,2.2,2.4,HICP
2026-03-31 11:00:00+02:00,0.000,0.00,0.100,0.100,2026-03-31,2.4,2.3,HICP


In [115]:
ois_pmi = pd.read_csv('data/input/PMI_composite_surprises.csv', index_col=2)
df_ois_pmi = pd.DataFrame()
for col in OIS_COLS:
    df_ois_pmi[col] = ois_pmi[ois_pmi['instrument_label']==col]['variation']

df_ois_pmi = df_ois_pmi.iloc[1::2]
df_ois_pmi['Date'] = pd.to_datetime(df_ois_pmi.index,format ='%Y-%m-%d %H:%M:%S%z', utc=True).date
df_ois_pmi['consensus'] = PMI_surprise_prelim['Srv Med'].astype(float).values
df_ois_pmi['actual'] = PMI_surprise_final['Srv Med'].astype(float).values
df_ois_pmi['indicator'] = 'PMI'
df_ois_pmi.round(2)

,OIS_1M,OIS_3M,OIS_6M,OIS_1Y,Date,consensus,actual,indicator
event_datetime,,,,,,,,
2013-08-05 10:00:00+02:00,0.00,0.00,0.10,0.10,2013-08-05,49.1,50.4,PMI
2013-09-04 10:00:00+02:00,0.20,-0.00,-0.00,-0.20,2013-09-04,50.9,51.7,PMI
2013-10-03 10:00:00+02:00,0.00,0.00,0.00,0.20,2013-10-03,51.8,52.1,PMI
2013-11-06 10:00:00+01:00,-0.00,0.00,0.10,0.20,2013-11-06,52.4,51.5,PMI
2013-12-04 10:00:00+01:00,0.00,-0.10,0.05,0.00,2013-12-04,52.0,51.5,PMI
...,...,...,...,...,...,...,...,...
2026-02-04 10:00:00+01:00,0.00,0.02,-0.04,-0.17,2026-02-04,51.9,51.5,PMI
2026-03-04 10:00:00+01:00,0.00,0.05,0.25,0.75,2026-03-04,51.5,51.9,PMI
2026-04-07 10:00:00+02:00,-0.02,-0.25,-0.60,-0.90,2026-04-07,51.0,50.5,PMI


In [116]:
df_surprise = pd.concat([df_ois,df_ois_pmi])
df_surprise = df_surprise.sort_index()
df_surprise

,OIS_1M,OIS_3M,OIS_6M,OIS_1Y,Date,consensus,actual,indicator
event_datetime,,,,,,,,
2013-07-31 11:00:00+02:00,0.000,-1.000000e-01,-1.500000e-01,-0.300,2013-07-31,1.2,1.1,HICP
2013-08-05 10:00:00+02:00,0.000,0.000000e+00,1.000000e-01,0.100,2013-08-05,49.1,50.4,PMI
2013-08-30 11:00:00+02:00,0.000,-2.000000e-01,0.000000e+00,0.100,2013-08-30,1.1,1.1,HICP
2013-09-04 10:00:00+02:00,0.200,-6.938894e-16,-6.938894e-16,-0.200,2013-09-04,50.9,51.7,PMI
2013-09-30 11:00:00+02:00,0.000,3.000000e-01,-4.000000e-01,-0.200,2013-09-30,1.1,1.0,HICP
...,...,...,...,...,...,...,...,...
2026-04-07 10:00:00+02:00,-0.015,-2.500000e-01,-6.000000e-01,-0.900,2026-04-07,51.0,50.5,PMI
2026-04-30 11:00:00+02:00,-0.235,-1.800000e-01,1.150000e-01,0.090,2026-04-30,2.2,2.2,HICP
2026-05-06 10:00:00+02:00,0.000,-4.850000e-01,-9.850000e-01,-1.820,2026-05-06,50.1,48.6,PMI


In [117]:
df_announcements = pd.DataFrame()
df_announcements['Date'] = df_surprise.index
df_announcements['indicator'] = df_surprise['indicator'].values
df_announcements['actual'] = df_surprise['actual'].values
df_announcements['consensus'] = df_surprise['consensus'].values
df_announcements

,Date,indicator,actual,consensus
0,2013-07-31 11:00:00+02:00,HICP,1.1,1.2
1,2013-08-05 10:00:00+02:00,PMI,50.4,49.1
2,2013-08-30 11:00:00+02:00,HICP,1.1,1.1
3,2013-09-04 10:00:00+02:00,PMI,51.7,50.9
4,2013-09-30 11:00:00+02:00,HICP,1.0,1.1
...,...,...,...,...
304,2026-04-07 10:00:00+02:00,PMI,50.5,51.0
305,2026-04-30 11:00:00+02:00,HICP,2.2,2.2
306,2026-05-06 10:00:00+02:00,PMI,48.6,50.1
307,2026-06-02 11:00:00+02:00,HICP,2.5,2.4


In [118]:
def keep_date_only(df):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date']).dt.normalize()
    return df


df_inflation = keep_date_only(df_inflation)
df_output = keep_date_only(df_output)
df_surprise = keep_date_only(df_surprise)

# Créer une clé Année-Mois (Period) pour chaque dataframe
def to_year_month(df):
    return set(df['Date'].dt.to_period('M'))

common_periods = (
    to_year_month(df_inflation)
    & to_year_month(df_output)
    & to_year_month(df_surprise)
)

# Filtrer chaque dataframe en gardant les lignes dont le mois est commun
df_inflation = df_inflation[df_inflation['Date'].dt.to_period('M').isin(common_periods)].copy()
df_output    = df_output[df_output['Date'].dt.to_period('M').isin(common_periods)].copy()
df_surprise  = df_surprise[df_surprise['Date'].dt.to_period('M').isin(common_periods)].copy()

In [ ]:
from GMM_estimation import run_all_maturities, run_all_maturities_daily

In [ ]:
res, ev = run_all_maturities(df_ois,df_ois_pmi,df_inflation,df_output)

--- Premiers événements (vérifier le mois de référence inféré) ---
indicator release_date   tau
     HICP   2013-07-31 24162
      PMI   2013-08-05 24162
     HICP   2013-08-30 24163
      PMI   2013-09-04 24163
     HICP   2013-09-30 24164
      PMI   2013-10-03 24164
     HICP   2013-10-31 24165
      PMI   2013-11-06 24165
--- Derniers événements ---
indicator release_date   tau
     HICP   2026-03-03 24313
      PMI   2026-03-04 24313
     HICP   2026-03-31 24314
      PMI   2026-04-07 24314
     HICP   2026-04-30 24315
      PMI   2026-05-06 24315
     HICP   2026-06-02 24316
      PMI   2026-06-03 24316

Nombre d'événements par indicateur :
indicator
PMI     155
HICP    154
Name: count, dtype: int64

=== Maturité 1 mois — T = 142 observations mensuelles ===


C:\Users\Z555495\AppData\Local\Temp\1\ipykernel_2196\3608209371.py:470: UserWarning: Les redémarrages convergent vers des objectifs différents (écart=0.04821) -> signe possible d'optima locaux ou d'identification faible de (beta, delta). Inspectez fit['restart_objectives'] avant d'interpréter les résultats.
  warnings.warn(


In [100]:
df_surprise_daily = build_df_surprise(df_announcements, ois_daily)

results, events = run_all_maturities_daily(df_surprise_daily, df_inflation, df_output)

--- Premiers événements (vérifier le mois de référence inféré) ---
indicator release_date   tau
     HICP   2013-07-31 24162
      PMI   2013-08-05 24162
     HICP   2013-08-30 24163
      PMI   2013-09-04 24163
     HICP   2013-09-30 24164
      PMI   2013-10-03 24164
     HICP   2013-10-31 24165
      PMI   2013-11-06 24165
--- Derniers événements ---
indicator release_date   tau
     HICP   2026-03-03 24313
      PMI   2026-03-04 24313
     HICP   2026-03-31 24314
      PMI   2026-04-07 24314
     HICP   2026-04-30 24315
      PMI   2026-05-06 24315
     HICP   2026-06-02 24316
      PMI   2026-06-03 24316

Nombre d'événements par indicateur :
indicator
PMI     155
HICP    154
Name: count, dtype: int64

=== Maturité 1M — T = 141 obs. ===


C:\Users\Z555495\AppData\Local\Temp\1\ipykernel_37580\3608209371.py:470: UserWarning: Les redémarrages convergent vers des objectifs différents (écart=0.03044) -> signe possible d'optima locaux ou d'identification faible de (beta, delta). Inspectez fit['restart_objectives'] avant d'interpréter les résultats.
  warnings.warn(
C:\Users\Z555495\AppData\Local\Temp\1\ipykernel_37580\3608209371.py:470: UserWarning: Les redémarrages convergent vers des objectifs différents (écart=0.08846) -> signe possible d'optima locaux ou d'identification faible de (beta, delta). Inspectez fit['restart_objectives'] avant d'interpréter les résultats.
  warnings.warn(


  Convergence : False | objectifs : [0.0293 0.0249 0.0463 0.0553 0.0277 0.0365]
  Empreinte HICP  (gamma_pi, gamma_y) = [ 9.4782658e-01 -3.8933274e+03]
  Empreinte PMI   (gamma_pi, gamma_y) = [3.81319317e-02 5.07151651e+03]
  beta_hat  =   -8.966  (se=4440.319, t=-0.00)
  delta_hat =    0.004  (se=10.071, t=0.00)
  J-test : J=4749.48, df=13, p=0.000
  Test restriction croisée : dJ=4549.97, p=0.000

=== Maturité 2M — T = 140 obs. ===


C:\Users\Z555495\AppData\Local\Temp\1\ipykernel_37580\3608209371.py:470: UserWarning: Les redémarrages convergent vers des objectifs différents (écart=0.09601) -> signe possible d'optima locaux ou d'identification faible de (beta, delta). Inspectez fit['restart_objectives'] avant d'interpréter les résultats.
  warnings.warn(
C:\Users\Z555495\AppData\Local\Temp\1\ipykernel_37580\3608209371.py:470: UserWarning: Les redémarrages convergent vers des objectifs différents (écart=0.6231) -> signe possible d'optima locaux ou d'identification faible de (beta, delta). Inspectez fit['restart_objectives'] avant d'interpréter les résultats.
  warnings.warn(


  Convergence : False | objectifs : [0.029  0.0207 0.0389 0.1167 0.0481 0.1073]
  Empreinte HICP  (gamma_pi, gamma_y) = [1.23568909e+00 4.73114737e+03]
  Empreinte PMI   (gamma_pi, gamma_y) = [2.49857745e-02 6.85932217e+03]
  beta_hat  =   68.607  (se=220.032, t=0.31)
  delta_hat =    0.349  (se=1.807, t=0.19)
  J-test : J=94.95, df=13, p=0.000
  Test restriction croisée : dJ=65.99, p=0.000

=== Maturité 3M — T = 139 obs. ===


...

C:\Users\Z555495\AppData\Local\Temp\1\ipykernel_37580\3608209371.py:470: UserWarning: Les redémarrages convergent vers des objectifs différents (écart=0.06526) -> signe possible d'optima locaux ou d'identification faible de (beta, delta). Inspectez fit['restart_objectives'] avant d'interpréter les résultats.
  warnings.warn(
C:\Users\Z555495\AppData\Local\Temp\1\ipykernel_37580\3608209371.py:470: UserWarning: Les redémarrages convergent vers des objectifs différents (écart=0.1049) -> signe possible d'optima locaux ou d'identification faible de (beta, delta). Inspectez fit['restart_objectives'] avant d'interpréter les résultats.
  warnings.warn(


  Convergence : False | objectifs : [0.0032 0.0685 0.0153 0.0298 0.0049 0.036 ]
  Empreinte HICP  (gamma_pi, gamma_y) = [2.03318236e+00 8.86477125e+03]
  Empreinte PMI   (gamma_pi, gamma_y) = [1.31349582e-01 3.08964691e+03]
  beta_hat  =    2.883  (se=0.166, t=17.35)
  delta_hat =   -0.001  (se=0.161, t=-0.01)
  J-test : J=4.14, df=13, p=0.990
  Test restriction croisée : dJ=0.46, p=0.500

=== Maturité 12M — T = 131 obs. ===


C:\Users\Z555495\AppData\Local\Temp\1\ipykernel_37580\3608209371.py:470: UserWarning: Les redémarrages convergent vers des objectifs différents (écart=0.02582) -> signe possible d'optima locaux ou d'identification faible de (beta, delta). Inspectez fit['restart_objectives'] avant d'interpréter les résultats.
  warnings.warn(


  Convergence : False | objectifs : [0.0043 0.0296 0.0158 0.0038 0.0094 0.0235]
  Empreinte HICP  (gamma_pi, gamma_y) = [  1.76586997 692.12369177]
  Empreinte PMI   (gamma_pi, gamma_y) = [1.18671859e-01 3.02366528e+03]
  beta_hat  =    8.396  (se=47.619, t=0.18)
  delta_hat =   -0.007  (se=1.552, t=-0.00)
  J-test : J=15.62, df=13, p=0.270
  Test restriction croisée : dJ=4.77, p=0.029


C:\Users\Z555495\AppData\Local\Temp\1\ipykernel_37580\3608209371.py:470: UserWarning: Les redémarrages convergent vers des objectifs différents (écart=0.06539) -> signe possible d'optima locaux ou d'identification faible de (beta, delta). Inspectez fit['restart_objectives'] avant d'interpréter les résultats.
  warnings.warn(


In [ ]:
from OLS_HPB import run_all_maturities_simple_daily

In [122]:
# df_surprise_daily = build_df_surprise(df_announcements, ois_daily)
results, events = run_all_maturities_simple_daily(
    df_surprise_daily, df_inflation, df_output,
    bootstrap=True, n_boot=300
)

--- Premiers événements (vérifier le mois de référence inféré) ---
indicator release_date   tau
     HICP   2013-07-31 24162
      PMI   2013-08-05 24162
     HICP   2013-08-30 24163
      PMI   2013-09-04 24163
     HICP   2013-09-30 24164
      PMI   2013-10-03 24164
     HICP   2013-10-31 24165
      PMI   2013-11-06 24165
--- Derniers événements ---
indicator release_date   tau
     HICP   2026-03-03 24313
      PMI   2026-03-04 24313
     HICP   2026-03-31 24314
      PMI   2026-04-07 24314
     HICP   2026-04-30 24315
      PMI   2026-05-06 24315
     HICP   2026-06-02 24316
      PMI   2026-06-03 24316

Nombre d'événements par indicateur :
indicator
PMI     155
HICP    154
Name: count, dtype: int64

=== [OLS simple, daily] Maturité 1M — T = 141 mois ===
  Empreintes (étape 1) :
    HICP  : gamma_pi=  0.9978  gamma_y= -0.0057
    PMI   : gamma_pi=  0.0269  gamma_y=  0.0061
  beta_hat  =    1.129  (se naïve=0.624, t=1.81)  ← SE INVALIDE, cf. bootstrap
  delta_hat =    4.377  (se n

In [ ]:
from Inflation_only import run_hicp_only

In [121]:
res, ev = run_hicp_only(df_surprise_daily,df_inflation, df_output)

--- Premiers événements (vérifier le mois de référence inféré) ---
indicator release_date   tau
     HICP   2013-07-31 24162
      PMI   2013-08-05 24162
     HICP   2013-08-30 24163
      PMI   2013-09-04 24163
     HICP   2013-09-30 24164
      PMI   2013-10-03 24164
     HICP   2013-10-31 24165
      PMI   2013-11-06 24165
--- Derniers événements ---
indicator release_date   tau
     HICP   2026-03-03 24313
      PMI   2026-03-04 24313
     HICP   2026-03-31 24314
      PMI   2026-04-07 24314
     HICP   2026-04-30 24315
      PMI   2026-05-06 24315
     HICP   2026-06-02 24316
      PMI   2026-06-03 24316

Nombre d'événements par indicateur :
indicator
PMI     155
HICP    154
Name: count, dtype: int64

=== [HICP-only] Maturité 1M — T = 142 obs. ===
  gamma_pi =   0.9542  (se=0.3248, t=2.94, R²=0.943)
  phi      =   1.1549  (se naïve=0.6682, t=1.73)  ← SE INVALIDE, cf. bootstrap
  beta (delta-method) =    1.210  (se≈0.700, t≈1.73)
  [bootstrap, 300/300 valides]
    se(phi)  = 23.536

In [63]:
"""
Version alignée sur 02_layerA_static_beta.py :
  - psi^(h)     = coeff de surprise dans d_ois ~ surprise    (bp/pp)
  - gamma_pi^(h)= coeff de surprise dans pi_{t+h} ~ surprise + pi_lag
  - beta^(h)    = psi^(h) / gamma_pi^(h)   [delta = 0 implicite]

Pas d'équation y, pas de delta séparé : on suppose que la réaction OIS
est intégralement attribuée à l'inflation (ou que delta*gamma_y est
absorbé dans psi et reste une limite de l'estimation baseline).
"""

# =========================================================================
# ÉTAPE 1a — psi^(h) : réaction OIS à la surprise (bp/pp)
# =========================================================================

def estimate_psi(panel):
    """
    d_ois[bp] ~ cst + psi * surprise_HICP
    surprise = actual - consensus
    Exact même spec que le code de référence.
    """
    surprise = panel["w_HICP"] - panel["wt_HICP"]
    X = sm.add_constant(pd.DataFrame({"surprise": surprise}, index=panel.index))
    return sm.OLS(panel["dois"], X).fit(
        cov_type="HAC", cov_kwds={"maxlags": NW_LAGS}
    )


# =========================================================================
# ÉTAPE 1b — gamma_pi^(h) : empreinte inflation
# =========================================================================

def estimate_gamma_pi(panel):
    """
    pi_{t+h} ~ cst + gamma_pi * surprise_HICP + pi_lag
    Même spec que le code de référence (ctrl=True).
    """
    surprise = panel["w_HICP"] - panel["wt_HICP"]
    X = sm.add_constant(pd.DataFrame({
        "surprise": surprise,
        "pi_lag":   panel["pilag_HICP"],
    }, index=panel.index))
    return sm.OLS(panel["pitgt"], X).fit(
        cov_type="HAC", cov_kwds={"maxlags": NW_LAGS}
    )


# =========================================================================
# ÉTAPE 2 — beta^(h) = psi^(h) / gamma_pi^(h), delta method
# =========================================================================

def compute_beta(psi, se_psi, gamma, se_gamma):
    """
    Delta method : Var(beta) = Var(psi)/gamma^2 + psi^2 * Var(gamma)/gamma^4
    Hypothèse : Cov(psi, gamma) = 0 (régressions sur des sorties disjointes).
    """
    if abs(gamma) < 1e-6:
        return np.nan, np.nan
    beta = psi / gamma
    var  = se_psi**2 / gamma**2 + psi**2 * se_gamma**2 / gamma**4
    return beta, np.sqrt(var)


# =========================================================================
# DRIVER
# =========================================================================

def estimate_layer_a(events, m, verbose=True):
    """
    Estimation HICP-only, séparation psi / gamma_pi propre.
    Pas d'équation y, pas de delta — identique à 02_layerA_static_beta.py.
    """
    panel = build_hicp_panel(events, m)  # inchangé

    if len(panel) < 10:
        return None

    model_psi   = estimate_psi(panel)
    model_gamma = estimate_gamma_pi(panel)

    psi      = model_psi.params["surprise"]
    se_psi   = model_psi.bse["surprise"]
    gamma    = model_gamma.params["surprise"]
    se_gamma = model_gamma.bse["surprise"]
    beta, se_beta = compute_beta(psi, se_psi, gamma, se_gamma)

    if verbose:
        t_psi = psi   / se_psi   if se_psi   > 0 else np.nan
        t_g   = gamma / se_gamma if se_gamma  > 0 else np.nan
        t_b   = beta  / se_beta  if se_beta   > 0 else np.nan
        print(f"\n=== [layer A] Maturité {m}M — T = {len(panel)} obs. ===")
        print(f"  psi      = {psi:8.3f} bp/pp  (se={se_psi:.3f}, t={t_psi:.2f})")
        print(f"  gamma_pi = {gamma:8.4f}       (se={se_gamma:.4f}, t={t_g:.2f}, "
              f"R²={model_gamma.rsquared:.3f})")
        print(f"  beta     = {beta:8.3f}        (se_dm={se_beta:.3f}, t={t_b:.2f})")

    return {
        "model_psi":   model_psi,
        "model_gamma": model_gamma,
        "psi":         psi,   "se_psi":   se_psi,
        "gamma_pi":    gamma, "se_gamma": se_gamma,
        "beta":        beta,  "se_beta":  se_beta,
        "panel":       panel,
    }


# =========================================================================
# BOOTSTRAP — SE correctes
# =========================================================================

def bootstrap_layer_a(events, m, n_boot=500, seed=0, verbose=True):
    panel_full = build_hicp_panel(events, m)
    months = panel_full.index.to_numpy()
    rng    = np.random.default_rng(seed)

    psis, betas = [], []
    for _ in range(n_boot):
        idx     = rng.choice(months, size=len(months), replace=True)
        panel_b = panel_full.loc[idx].reset_index(drop=True)
        try:
            mp = estimate_psi(panel_b)
            mg = estimate_gamma_pi(panel_b)
            p  = mp.params["surprise"]
            g  = mg.params["surprise"]
            psis.append(p)
            if abs(g) > 1e-6:
                betas.append(p / g)
        except Exception:
            continue

    psis  = np.array(psis)
    betas = np.array(betas)
    result = {
        "psi_se":    psis.std(ddof=1),
        "psi_ci90":  np.percentile(psis,  [5, 95]),
        "beta_se":   betas.std(ddof=1),
        "beta_ci90": np.percentile(betas, [5, 95]),
        "n_valid":   len(psis),
    }
    if verbose:
        print(f"  [bootstrap {result['n_valid']}/{n_boot}]  "
              f"se(psi)={result['psi_se']:.3f}  "
              f"se(beta)={result['beta_se']:.3f}  "
              f"IC90(beta)={np.round(result['beta_ci90'], 3)}")
    return result


# =========================================================================
# POINT D'ENTRÉE
# =========================================================================

def run_layer_a(df_surprise_daily, df_inflation, df_output,
                bootstrap=True, n_boot=300):
    events = build_event_table_daily(df_surprise_daily, df_inflation, df_output)
    inspect_reference_months(events)

    results = {}
    for m in HORIZONS_M:
        fit = estimate_layer_a(events, m)
        if fit is None:
            continue
        if bootstrap:
            fit["bootstrap"] = bootstrap_layer_a(events, m, n_boot=n_boot)
        results[m] = fit
    return results, events

In [103]:
res, evs = run_layer_a(df_surprise_daily, df_inflation, df_output)

--- Premiers événements (vérifier le mois de référence inféré) ---
indicator release_date   tau
     HICP   2013-07-31 24162
      PMI   2013-08-05 24162
     HICP   2013-08-30 24163
      PMI   2013-09-04 24163
     HICP   2013-09-30 24164
      PMI   2013-10-03 24164
     HICP   2013-10-31 24165
      PMI   2013-11-06 24165
--- Derniers événements ---
indicator release_date   tau
     HICP   2026-03-03 24313
      PMI   2026-03-04 24313
     HICP   2026-03-31 24314
      PMI   2026-04-07 24314
     HICP   2026-04-30 24315
      PMI   2026-05-06 24315
     HICP   2026-06-02 24316
      PMI   2026-06-03 24316

Nombre d'événements par indicateur :
indicator
PMI     155
HICP    154
Name: count, dtype: int64

=== [layer A] Maturité 1M — T = 142 obs. ===
  psi      =    1.102 bp/pp  (se=0.638, t=1.73)
  gamma_pi =   1.0356       (se=0.4416, t=2.34, R²=0.932)
  beta     =    1.064        (se_dm=0.765, t=1.39)
  [bootstrap 300/300]  se(psi)=0.628  se(beta)=2.835  IC90(beta)=[0.129 4.382]

==

In [ ]:
from ortho_OLS import run_ols_orth

In [119]:
res, ev = run_ols_orth(df_surprise_daily,df_inflation, df_output)

--- Premiers événements (vérifier le mois de référence inféré) ---
indicator release_date   tau
     HICP   2013-07-31 24162
      PMI   2013-08-05 24162
     HICP   2013-08-30 24163
      PMI   2013-09-04 24163
     HICP   2013-09-30 24164
      PMI   2013-10-03 24164
     HICP   2013-10-31 24165
      PMI   2013-11-06 24165
--- Derniers événements ---
indicator release_date   tau
     HICP   2026-03-03 24313
      PMI   2026-03-04 24313
     HICP   2026-03-31 24314
      PMI   2026-04-07 24314
     HICP   2026-04-30 24315
      PMI   2026-05-06 24315
     HICP   2026-06-02 24316
      PMI   2026-06-03 24316

Nombre d'événements par indicateur :
indicator
PMI     155
HICP    154
Name: count, dtype: int64

=== [OLS+orth] Maturité 1M — T = 141 mois ===
  Corrélation brute HICP/PMI = 0.115  R²(PMI ~ HICP) = 0.013  [corrélation résiduelle ≈ 0 par construction]
  gamma_pi : HICP=  1.0366  PMI_orth=  0.0472  (R²=0.933)
  gamma_y  : HICP= -0.0155  PMI_orth=  0.0095  (R²=0.421)
  det(Γ) =    